In [1]:
%pip install scikit-learn numpy pandas nltk scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 37.0 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 41.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 40.6 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 30.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 38.8 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.2/807.2 kB 24.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [nltk]2m11/12 [nltk]t-learn]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk
import numpy as np
import pandas as pd
import time

from nltk.corpus import movie_reviews

from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer
)

from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
nltk.download("movie_reviews")

[nltk_data] Downloading package movie_reviews to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.


True

In [4]:
documents = [
    (movie_reviews.raw(fileid), category)
    for category in movie_reviews.categories()
    for fileid in movie_reviews.fileids(category)
]

df = pd.DataFrame(documents, columns=["text", "sentiment"])

print("Dataset shape:", df.shape)
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

print("\nFirst review:")
print(df.iloc[0]["text"][:500])

print("\nFirst review's sentiment:", df.iloc[0]["sentiment"])

Dataset shape: (2000, 2)

Sentiment distribution:
sentiment
neg    1000
pos    1000
Name: count, dtype: int64

First review:
plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares . 
what's the deal ? 
watch the movie and " sorta " find out . . . 
critique : a mind-fuck movie for the teen generation that touches on a very cool idea , but presents it in a very bad package . 
which is what makes this review an even harder one to write , since i generally applaud films which attempt

First review's sentiment: neg


In [5]:
X = df["text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 1600
Testing reviews: 400


In [6]:
count_vectorizer = CountVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_count = count_vectorizer.fit_transform(X_train)
X_test_count = count_vectorizer.transform(X_test)

print("Training feature matrix:", X_train_count.shape)
print("Testing feature matrix:", X_test_count.shape)
print("\nSample vocabulary:")
print(count_vectorizer.get_feature_names_out()[:20])

Training feature matrix: (1600, 5000)
Testing feature matrix: (400, 5000)

Sample vocabulary:
['000' '10' '100' '101' '11' '12' '13' '14' '15' '16' '17' '18' '1900'
 '1960' '1960s' '1970s' '1980' '1980s' '1981' '1984']


In [7]:
nb_model = MultinomialNB()

start_time = time.time()
nb_model.fit(X_train_count, y_train)
nb_training_time = time.time() - start_time

print("Naive Bayes training completed!")
print(f"Training time: {nb_training_time:.4f} seconds")

Naive Bayes training completed!
Training time: 0.0068 seconds


In [8]:
nb_predictions = nb_model.predict(X_test_count)

nb_accuracy = accuracy_score(y_test, nb_predictions)

print(f"Naive Bayes accuracy: {nb_accuracy:.4f}")
print("\nClassification report:")
print(classification_report(y_test, nb_predictions))

Naive Bayes accuracy: 0.8050

Classification report:
              precision    recall  f1-score   support

         neg       0.79      0.82      0.81       200
         pos       0.82      0.79      0.80       200

    accuracy                           0.81       400
   macro avg       0.81      0.80      0.80       400
weighted avg       0.81      0.81      0.80       400



## Task 2: Understanding CountVectorizer

### Question 1: Explain the role of CountVectorizer in text classification. How does it prepare text data for machine learning?

CountVectorizer converts text documents into numerical feature vectors by building a vocabulary of words and counting how often each word appears in each document. Each vocabulary word becomes a feature, allowing a machine learning model to process textual information mathematically. In this assignment, CountVectorizer represents movie reviews using word counts, with a maximum vocabulary size of 5,000 features. These numerical representations help the Naive Bayes classifier identify patterns associated with positive and negative sentiment.

### Question 2: What is the purpose of train_test_split? Why is it important in machine learning evaluation?

The `train_test_split` function divides a dataset into separate training and testing subsets. The training subset is used to fit the machine learning model, while the testing subset evaluates its performance on unseen examples. In this assignment, 80% of the movie reviews are used for training and 20% for testing. This separation helps assess whether the model generalizes beyond its training data rather than simply memorizing examples. Using `random_state=42` also makes the split reproducible, while stratification preserves class proportions.

### Question 3: Analyze the accuracy calculation in the Naive Bayes classifier. What does the score tell us about model performance?

Accuracy is calculated by dividing the number of correctly classified test reviews by the total number of test reviews. For example, an accuracy of 0.80 means the model correctly predicts the sentiment of 80% of the test reviews. Since this dataset contains equal numbers of positive and negative reviews, accuracy provides a useful overall measure of performance. However, it does not reveal every type of classification error. Precision, recall, and F1-score provide additional information about how effectively the classifier identifies each sentiment class.

In [9]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF training matrix:", X_train_tfidf.shape)
print("TF-IDF testing matrix:", X_test_tfidf.shape)

TF-IDF training matrix: (1600, 5000)
TF-IDF testing matrix: (400, 5000)


In [10]:
tfidf_nb_model = MultinomialNB()

start_time = time.time()
tfidf_nb_model.fit(X_train_tfidf, y_train)
tfidf_nb_training_time = time.time() - start_time

tfidf_nb_predictions = tfidf_nb_model.predict(X_test_tfidf)

tfidf_nb_accuracy = accuracy_score(
    y_test,
    tfidf_nb_predictions
)

print(f"TF-IDF Naive Bayes accuracy: {tfidf_nb_accuracy:.4f}")
print(f"Training time: {tfidf_nb_training_time:.4f} seconds")

print("\nClassification report:")
print(classification_report(y_test, tfidf_nb_predictions))

TF-IDF Naive Bayes accuracy: 0.8075
Training time: 0.0073 seconds

Classification report:
              precision    recall  f1-score   support

         neg       0.79      0.83      0.81       200
         pos       0.82      0.79      0.80       200

    accuracy                           0.81       400
   macro avg       0.81      0.81      0.81       400
weighted avg       0.81      0.81      0.81       400



In [11]:
nb_comparison = pd.DataFrame({
    "Method": [
        "CountVectorizer + Naive Bayes",
        "TF-IDF + Naive Bayes"
    ],
    "Accuracy": [
        nb_accuracy,
        tfidf_nb_accuracy
    ],
    "Training Time (seconds)": [
        nb_training_time,
        tfidf_nb_training_time
    ]
})

print(nb_comparison.to_string(index=False))

                       Method  Accuracy  Training Time (seconds)
CountVectorizer + Naive Bayes    0.8050                 0.006790
         TF-IDF + Naive Bayes    0.8075                 0.007288


In [12]:
accuracy_difference = tfidf_nb_accuracy - nb_accuracy

print(f"Accuracy difference: {accuracy_difference:+.4f}")
print(f"Difference in percentage points: {accuracy_difference * 100:+.2f}")

Accuracy difference: +0.0025
Difference in percentage points: +0.25


## Task 3: Advanced Feature Engineering

### Comparison and Interpretation

CountVectorizer represents each review using word occurrence counts, whereas TfidfVectorizer weights terms according to their frequency within a review and their distribution across the collection of reviews. Both methods convert text into numerical features that can be processed by Naive Bayes.

In my experiment, CountVectorizer achieved an accuracy of **[insert your CountVectorizer accuracy]**, while TF-IDF achieved **[insert your TF-IDF accuracy]**. The difference was **[insert the difference in percentage points]**. These results show which representation performed better on my test set.

TF-IDF may improve performance by reducing the influence of very common words and emphasizing more discriminative terms. However, raw word counts can also work well for Naive Bayes because word occurrence frequencies are directly relevant to its probabilistic model. Therefore, the better method depends on the dataset and the observed results.

### Conclusion

The comparison demonstrates that feature representation can influence classification performance. I used the same train-test split and Naive Bayes classifier for both methods to make the comparison more consistent. I based my conclusion on the accuracy scores obtained from my notebook rather than assuming that one approach must always perform better.

In [13]:
svm_model = LinearSVC(random_state=42, max_iter=5000)

start_time = time.time()
svm_model.fit(X_train_count, y_train)
svm_training_time = time.time() - start_time

print("SVM training completed!")
print(f"Training time: {svm_training_time:.4f} seconds")

SVM training completed!
Training time: 0.1892 seconds


In [14]:
svm_predictions = svm_model.predict(X_test_count)

svm_accuracy = accuracy_score(
    y_test,
    svm_predictions
)

print(f"SVM accuracy: {svm_accuracy:.4f}")
print("\nClassification report:")
print(classification_report(y_test, svm_predictions))

SVM accuracy: 0.8350

Classification report:
              precision    recall  f1-score   support

         neg       0.83      0.83      0.83       200
         pos       0.83      0.83      0.83       200

    accuracy                           0.83       400
   macro avg       0.83      0.83      0.83       400
weighted avg       0.83      0.83      0.83       400



In [15]:
comparison_df = pd.DataFrame({
    "Model": [
        "CountVectorizer + Naive Bayes",
        "TF-IDF + Naive Bayes",
        "CountVectorizer + LinearSVC"
    ],
    "Accuracy": [
        nb_accuracy,
        tfidf_nb_accuracy,
        svm_accuracy
    ],
    "Training Time (seconds)": [
        nb_training_time,
        tfidf_nb_training_time,
        svm_training_time
    ]
})

comparison_df["Accuracy (%)"] = (
    comparison_df["Accuracy"] * 100
)

comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print(comparison_df.to_string(index=False))

                        Model  Accuracy  Training Time (seconds)  Accuracy (%)
  CountVectorizer + LinearSVC    0.8350                 0.189196         83.50
         TF-IDF + Naive Bayes    0.8075                 0.007288         80.75
CountVectorizer + Naive Bayes    0.8050                 0.006790         80.50


## Task 4: Alternative Models

### Naive Bayes and SVM Comparison

The Naive Bayes classifier and Linear Support Vector Classifier (LinearSVC) were evaluated using the same CountVectorizer features and train-test split. Naive Bayes uses probability estimates and assumes conditional independence between features given the class. LinearSVC instead learns a linear decision boundary that separates positive and negative reviews.

In my experiment, CountVectorizer with Naive Bayes achieved an accuracy of **[insert your Naive Bayes accuracy]**, while LinearSVC achieved **[insert your SVM accuracy]**. The faster model was **[insert the model with the lower observed training time]**, based on the measured training times in my notebook.

Naive Bayes is generally simple and computationally efficient, making it a useful baseline for text classification. LinearSVC can perform well with high-dimensional, sparse text features, but it requires optimization and may produce convergence warnings if the iteration limit is insufficient.

### Conclusion and Trade-offs

The comparison shows that model selection should consider accuracy, training time, and complexity rather than accuracy alone. In my experiment, **[identify the model with the higher observed accuracy]** performed better in terms of accuracy. These findings apply to the selected dataset, feature representation, and train-test split; other settings may produce different results.